In [0]:
# filename: notebooks/main_notebook.py
# ─────────────────────────────────────────────────────────────────────
#  PURPOSE: Orchestrates the full pipeline — one notebook to run
#  FLOW:    Install → Import → Secrets → Generate JSON → Upload
# ─────────────────────────────────────────────────────────────────────

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 1 — Install packages (run first, only once per session)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
%pip install azure-storage-file-datalake

In [0]:
dbutils.library.restartPython()


In [0]:
# Code Generated by Sidekick is for learning and experimentation purposes only.
import sys
import os

PROJECT_ROOT = "/Workspace/Users/anujjain5699@gmail.com/data-engineer-practice/stream_table_ingestion"

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"✅ Project root set: {PROJECT_ROOT}")
print(f"\n📁 Files Python can see:\n")
for folder in ["config", "utils"]:
    print(f"📁 {folder}/")
    path = os.path.join(PROJECT_ROOT, folder)
    for f in sorted(os.listdir(path)):
        size   = os.path.getsize(os.path.join(path, f))
        ftype  = "✅ real .py" if f.endswith(".py") else "📓 notebook"
        print(f"   └── {f:45s} {size:5d} bytes  {ftype}")


In [0]:
# Code Generated by Sidekick is for learning and experimentation purposes only.
from config.settings import SECRET_SCOPE

print(f"🔍 Testing access to scope: {SECRET_SCOPE}\n")

try:
    secrets = dbutils.secrets.list(SECRET_SCOPE)
    print("✅ Permission granted! Secrets available:\n")
    for s in secrets:
        print(f"   └── {s.key}")
except Exception as e:
    print(f"❌ Still blocked: {e}")
    print("\n⏳ Wait 5 more minutes and try again")


In [0]:
# Code Generated by Sidekick is for learning and experimentation purposes only.
from config.settings      import LOCAL_FILE_PATH
from utils.secret_manager import get_all_secrets
from utils.adls_client    import get_adls_client, upload_file
from utils.json_generator import save_orders_json

print("✅ All modules imported successfully!")


In [0]:
# Code Generated by Sidekick is for learning and experimentation purposes only.
print("=" * 60)
print("   ADLS GEN2 SECURE UPLOAD PIPELINE — SERVERLESS")
print("=" * 60)

# Step 1 — Fetch secrets from Key Vault via Secret Scope
secrets = get_all_secrets()

# Step 2 — Generate sample JSON file locally
save_orders_json(
    file_path   = LOCAL_FILE_PATH,
    num_records = 5
)

# Step 3 — Connect to ADLS Gen2
adls_client = get_adls_client(
    account_name = secrets["account_name"],
    account_key  = secrets["storage_key"]
)

# Step 4 — Upload file to ADLS Gen2
streaming_path = upload_file(
    client          = adls_client,
    container_name  = secrets["container_name"],
    local_file_path = LOCAL_FILE_PATH
)

print(f"\n{'=' * 60}")
print(f"  ✅ PIPELINE COMPLETE!")
print(f"{'=' * 60}")
print(f"  📍 Streaming path: {streaming_path}")
print(f"{'=' * 60}")


In [0]:
# Code Generated by Sidekick is for learning and experimentation purposes only.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Verify Secret Scope and secrets exist
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from config.settings import (
    SECRET_SCOPE,
    SECRET_KEY_STORAGE_KEY,
    SECRET_KEY_ACCOUNT_NAME,
    SECRET_KEY_CONTAINER_NAME
)

print("🔍 Checking Secret Scope setup...\n")

# CHECK 1 — List all available scopes
print("📋 Available Secret Scopes on this cluster:")
try:
    scopes = [s.name for s in dbutils.secrets.listScopes()]
    for s in scopes:
        print(f"   └── {s}")
    if SECRET_SCOPE in scopes:
        print(f"\n   ✅ '{SECRET_SCOPE}' scope EXISTS")
    else:
        print(f"\n   ❌ '{SECRET_SCOPE}' scope NOT FOUND")
        print(f"      Scopes in settings.py: SECRET_SCOPE = '{SECRET_SCOPE}'")
        print(f"      Must match exactly — check spelling!")
except Exception as e:
    print(f"   ❌ Cannot list scopes: {e}")

# CHECK 2 — List secrets inside the scope
print(f"\n📋 Secrets inside scope '{SECRET_SCOPE}':")
try:
    secret_keys = [s.key for s in dbutils.secrets.list(SECRET_SCOPE)]
    for k in secret_keys:
        print(f"   └── {k}")

    # Check each required secret exists
    print(f"\n🔬 Matching against settings.py values:")
    for label, value in {
        "SECRET_KEY_STORAGE_KEY"   : SECRET_KEY_STORAGE_KEY,
        "SECRET_KEY_ACCOUNT_NAME"  : SECRET_KEY_ACCOUNT_NAME,
        "SECRET_KEY_CONTAINER_NAME": SECRET_KEY_CONTAINER_NAME
    }.items():
        status = "✅ FOUND" if value in secret_keys else "❌ NOT FOUND"
        print(f"   {status} → {label} = '{value}'")

except Exception as e:
    print(f"   ❌ Cannot list secrets: {e}")
